# 8.2 Principal Component Analysis (PCA)

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 8: Dimensionality Reduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-08-dimensionality-reduction/08.02_principal-component-analysis.ipynb)

Companion code for Section 8.2 of the book. Run the cells in order from top to bottom.

**Example 8.2.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

class PCAScratch:
    '''PCA from scratch using eigendecomposition of covariance matrix.'''

    def __init__(self, n_components=None):
        self.n_components = n_components
        self.components_  = None   # Principal components (eigenvectors)
        self.mean_        = None   # Feature means
        self.explained_variance_          = None
        self.explained_variance_ratio_    = None
        self.cumulative_variance_ratio_   = None

    def fit(self, X):
        n, d = X.shape

        # Step 1: Centre the data
        self.mean_ = X.mean(axis=0)
        X_centred  = X - self.mean_

        # Step 2: Covariance matrix
        C = (1/(n-1)) * X_centred.T @ X_centred   # (d × d)

        # Step 3: Eigendecomposition
        eigenvalues, eigenvectors = np.linalg.eigh(C)  # eigh for symmetric

        # Sort descending (eigh returns ascending order)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues  = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Store
        k = self.n_components or d
        self.components_ = eigenvectors[:, :k].T   # Shape: (k, d)
        self.explained_variance_ = eigenvalues[:k]
        total_var = eigenvalues.sum()
        self.explained_variance_ratio_ = eigenvalues[:k] / total_var
        self.cumulative_variance_ratio_ = np.cumsum(self.explained_variance_ratio_)

        return self

    def transform(self, X):
        '''Project data onto principal components.'''
        X_centred = X - self.mean_
        return X_centred @ self.components_.T   # Shape: (n, k)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        '''Reconstruct data from principal component scores.'''
        return Z @ self.components_ + self.mean_

    def reconstruction_error(self, X):
        '''Mean squared reconstruction error.'''
        Z    = self.transform(X)
        X_hat= self.inverse_transform(Z)
        return np.mean((X - X_hat)**2)


# ── Apply to Breast Cancer dataset ────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

# Always scale before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Full PCA (keep all components)
pca_full = PCAScratch(n_components=None)
pca_full.fit(X_scaled)

print('=== PCA from Scratch - Breast Cancer (30 features) ===')
print(f'\n{"PC":>4} {"Eigenvalue":>12} {"Var Explained":>15} {"Cumulative":>12}')
print('='*47)
for i, (ev, evr, cum) in enumerate(zip(
    pca_full.explained_variance_[:10],
    pca_full.explained_variance_ratio_[:10],
    pca_full.cumulative_variance_ratio_[:10]
)):
    marker = ' ← 90%' if abs(cum - 0.9) < 0.05 else ''
    print(f'{i+1:>4} {ev:>12.4f} {evr:>15.4f} {cum:>12.4f}{marker}')

# How many components needed?
for target_var in [0.80, 0.85, 0.90, 0.95, 0.99]:
    n_comp = np.argmax(pca_full.cumulative_variance_ratio_ >= target_var) + 1
    print(f'  {target_var:.0%} variance: {n_comp} components')

# ── 2D Projection ─────────────────────────────────────────────────
pca_2d = PCAScratch(n_components=2)
Z_2d   = pca_2d.fit_transform(X_scaled)

print(f'\nPC1 explained: {pca_2d.explained_variance_ratio_[0]:.1%}')
print(f'PC2 explained: {pca_2d.explained_variance_ratio_[1]:.1%}')
print(f'Total:         {pca_2d.cumulative_variance_ratio_[1]:.1%}')

# ── Reconstruction error vs number of components ──────────────────
print('\nReconstruction error vs n_components:')
for k in [1, 2, 3, 5, 10, 15, 20, 25, 30]:
    pca_k = PCAScratch(n_components=k)
    pca_k.fit(X_scaled)
    err = pca_k.reconstruction_error(X_scaled)
    cum_var = pca_k.cumulative_variance_ratio_[-1]
    print(f'  k={k:>2}: Recon Error={err:.6f}  Var Retained={cum_var:.1%}')

# ── Compare with sklearn ──────────────────────────────────────────
from sklearn.decomposition import PCA
pca_sk = PCA(n_components=2)
Z_sk   = pca_sk.fit_transform(X_scaled)
print(f'\nSklearn PC1 var: {pca_sk.explained_variance_ratio_[0]:.4f}')
print(f'Scratch PC1 var: {pca_2d.explained_variance_ratio_[0]:.4f}')
print(f'Match (within sign flip): {np.allclose(np.abs(Z_2d), np.abs(Z_sk), rtol=1e-3)}')

**Example 8.2.2**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()
X_sc = StandardScaler().fit_transform(data.data)
y    = data.target

pca = PCA()
Z   = pca.fit_transform(X_sc)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('PCA Analysis - Breast Cancer Dataset (30 Features → 2D)',
            fontsize=14, fontweight='bold')

# 1. Scree plot (variance explained per component)
ax = axes[0,0]
n_show = 15
ax.bar(range(1,n_show+1), pca.explained_variance_ratio_[:n_show],
      color='#1976D2', alpha=0.8, edgecolor='white')
ax.plot(range(1,n_show+1), pca.explained_variance_ratio_[:n_show],
       'ro-', ms=6, lw=2)
ax.set_xlabel('Principal Component'); ax.set_ylabel('Variance Explained')
ax.set_title('Scree Plot - Individual Variance per PC')
ax.grid(alpha=0.3)

# 2. Cumulative variance explained
ax = axes[0,1]
cum_var = pca.explained_variance_ratio_.cumsum()
ax.plot(range(1,31), cum_var, 'b-o', ms=6, lw=2.5)
for threshold in [0.80, 0.90, 0.95, 0.99]:
    k = np.argmax(cum_var >= threshold) + 1
    ax.axhline(threshold, color='gray', ls='--', alpha=0.5)
    ax.axvline(k, color='gray', ls=':', alpha=0.5)
    ax.annotate(f'{threshold:.0%}→k={k}', (k+0.3, threshold-0.02), fontsize=9)
ax.set_xlabel('Number of Components'); ax.set_ylabel('Cumulative Variance')
ax.set_title('Cumulative Variance Explained')
ax.grid(alpha=0.3)

# 3. 2D scatter coloured by class
ax = axes[0,2]
colors = ['#E53935' if c==0 else '#1976D2' for c in y]
ax.scatter(Z[:,0], Z[:,1], c=colors, s=20, alpha=0.7, edgecolors='none')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#E53935',label='Malignant'),
                   Patch(color='#1976D2',label='Benign')], fontsize=10)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA 2D Projection - Classes Visible!'); ax.grid(alpha=0.3)

# 4. Loadings heatmap (which features contribute to each PC?)
ax = axes[1,0]
loadings = pd.DataFrame(pca.components_[:5,:],
                         columns=data.feature_names,
                         index=[f'PC{i+1}' for i in range(5)])
sns.heatmap(loadings, ax=ax, cmap='RdYlBu_r', center=0,
           annot=False, linewidths=0.3, cbar_kws={'shrink':0.8})
ax.set_title('PC Loadings (top 5 PCs)\n(colour = contribution direction+magnitude)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)

# 5. PC1 loadings bar chart
ax = axes[1,1]
pc1_loadings = pca.components_[0]
sorted_idx = np.argsort(np.abs(pc1_loadings))[::-1][:15]
colors_bar = ['#E53935' if v<0 else '#1976D2' for v in pc1_loadings[sorted_idx]]
ax.barh(range(15), pc1_loadings[sorted_idx][::-1], color=colors_bar[::-1],
       edgecolor='white')
ax.set_yticks(range(15))
ax.set_yticklabels([data.feature_names[i] for i in sorted_idx[::-1]], fontsize=8)
ax.set_xlabel('PC1 Loading (contribution)'); ax.axvline(0, color='black', lw=1)
ax.set_title('PC1 Feature Loadings\n(top 15 contributing features)')
ax.grid(alpha=0.3, axis='x')

# 6. Reconstruction quality
ax = axes[1,2]
k_values = range(1, 31)
errors = []
for k in k_values:
    pca_k = PCA(n_components=k)
    Z_k   = pca_k.fit_transform(X_sc)
    X_rec = pca_k.inverse_transform(Z_k)
    errors.append(np.mean((X_sc - X_rec)**2))

ax2 = ax.twinx()
l1, = ax.plot(k_values, errors, 'r-o', ms=5, lw=2, label='Reconstruction Error')
l2, = ax2.plot(k_values, np.cumsum(pca.explained_variance_ratio_), 'b-s',
              ms=5, lw=2, label='Cumulative Var')
ax.set_xlabel('Number of Components'); ax.set_ylabel('MSE', color='red')
ax2.set_ylabel('Cumulative Variance', color='blue')
ax.set_title('Reconstruction Error vs k')
ax.legend([l1,l2], ['Recon Error','Cum Var'], loc='center right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── PCA as preprocessing for ML ──────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

print('=== PCA as ML Preprocessing ===')
print(f'{"Components":>12} {"CV Accuracy":>14} {"Features":>10} {"Compression":>12}')
print('='*52)
for k in [2, 5, 10, 15, 20, 30]:
    pipe = Pipeline([('pca', PCA(n_components=k)), ('lr', LogisticRegression(max_iter=1000))])
    scores = cross_val_score(pipe, X_sc, y, cv=5, scoring='accuracy')
    print(f'{k:>12} {scores.mean():>14.4f} {k:>10}/{30}  {(1-k/30)*100:>10.1f}% reduced')